In [10]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from google.colab import files
from IPython.core import history
from os import wait

In [6]:
canvas_height = 512
canvas_width = 700
canvas = np.ones((canvas_height, canvas_width, 3), dtype=np.uint8) * 255

brush_color = (0, 0, 0)
brush_size = 10
brush_alpha = 1.0
strokes = []
history = []

In [7]:
def draw_stroke(points, color_bgr, size, alpha=1.0):
  global canvas
  if len(points) == 0:
    return

  history.append(canvas.copy())

  overlay = canvas.copy()
  mask = np.zeros((canvas_height, canvas_width), dtype=np.uint8)

  if len(points) == 1:
    x, y = points[0]
    cv2.circle(overlay, (x, y), size, color_bgr, -1)
    cv2.circle(mask, (x, y), size, 255, -1)
  else:
    for start, end in zip(points[:-1], points[1:]):
      cv2.line(overlay, start, end, color_bgr, size * 2, cv2.LINE_AA)
      cv2.line(mask, start, end, 255, size * 2, cv2.LINE_AA)
    for x, y in points:
      cv2.circle(overlay, (x, y), size, color_bgr, -1)
      cv2.circle(mask, (x, y), size, 255, -1)

  blended = cv2.addWeighted(canvas, 1.0 - alpha, overlay, alpha, 0.0)
  canvas[mask > 0] = blended[mask > 0]

In [18]:
red_trackbar = widgets.IntSlider(min=0, max=255, value=0, step=1, description='Red')
green_trackbar = widgets.IntSlider(min=0, max=255, value=0, step=1, description='Green')
blue_trackbar = widgets.IntSlider(min=0, max=255, value=0, step=1, description='Blue')
size_trackbar = widgets.IntSlider(min=1, max=40, step=1, value=10, description='Size')

opacity_trackbar = widgets.IntSlider(min=0, max=100, value=100, step=1, description='Opacity %')

x_trackbar = widgets.IntSlider(min=0, max=canvas_width-1, value=100, step=1, description='X')
y_trackbar = widgets.IntSlider(min=0, max=canvas_height-1, value=100, step=1, description='Y')

draw_point_button = widgets.Button(description='Draw Point', button_style='primary')
sample_stroke_button = widgets.Button(description='Add Sample Stroke', button_style='success')
eraser_button = widgets.Button(description='Erase', button_style='warning')
clear_button = widgets.Button(description='Clear')
undo_button = widgets.Button(description='Undo')
save_button = widgets.Button(description='Save PNG', button_style='info')
status = widgets.HTML()
output_area = widgets.Output()

def selected_color_bgr():
  return (blue_trackbar.value, green_trackbar.value, red_trackbar.value)

def refresh_canvas():
  global brush_color, brush_alpha, brush_size
  brush_color = selected_color_bgr()
  brush_alpha = opacity_trackbar.value / 100.0
  brush_size = size_trackbar.value
  rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)

  with output_area:
    output_area.clear_output(wait=True)
    plt.figure(figsize=(11, 7))
    plt.imshow(rgb)
    plt.title(
        f'Paint | R={red_trackbar.value}, G={green_trackbar.value}, '
            f'B={blue_trackbar.value}, Size={brush_size}, Opacity={opacity_trackbar.value}%'
        )
    plt.xlim([0, canvas_width])
    plt.ylim([canvas_height, 0])
    plt.grid(alpha=0.25)
    plt.show()
  history_len_str = str(len(globals()['history'])) if isinstance(globals()['history'], list) else 'Error'
  status.value = (
      f'<b>RGB:</b> ({red_trackbar.value}, {green_trackbar.value}, {blue_trackbar.value}) '
      f'&nbsp; <b>Size:</b> {brush_size} '
      f'&nbsp; <b>Opacity:</b> {opacity_trackbar.value}% '
      f'&nbsp; <b>History:</b> {history_len_str}'
  )

def add_stroke(points, color=None, alpha=None):
  if color is None:
    color = selected_color_bgr()
  if alpha is None:
    alpha = opacity_trackbar.value / 100.0
  strokes.append({'points': points, 'color_bgr': color, 'size': size_trackbar.value, 'opacity': alpha})
  draw_stroke(points, color, size_trackbar.value, alpha)
  refresh_canvas()

def on_draw_point(button):
  add_stroke([(x_trackbar.value, y_trackbar.value)])

def on_sample_stroke(button):
  x, y = x_trackbar.value, y_trackbar.value
  points = [
      (max(0, x - 80), y),
      (max(0, x - 40), max(0, y - 30)),
      (x, y),
      (min(canvas_width - 1, x + 40), min(canvas_height - 1, y + 30)),
      (min(canvas_width - 1, x + 80), y)
  ]
  add_stroke(points)

def on_eraser(button):
  x, y = x_trackbar.value, y_trackbar.value
  points = [(max(0, x - 80), y), (x, y), (min(canvas_width - 1, x + 80), y)]
  add_stroke(points, color=(255, 255, 255), alpha=1.0)

def on_undo(button):
  global canvas, strokes
  if globals()['history'] and isinstance(globals()['history'], list):
    canvas = globals()['history'].pop()
    if strokes:
      strokes.pop()
  refresh_canvas()

def on_save(button):
  filename = 'paint_output.png'
  cv2.imwrite(filename, canvas)
  files.download(filename)
  status.value = f'<b>Saved:</b> {filename}'

def on_clear(button):
  global canvas, strokes
  if isinstance(globals()['history'], list):
      globals()['history'].append(canvas.copy())
  canvas = np.ones((canvas_height, canvas_width, 3), dtype=np.uint8) * 255
  strokes = []
  refresh_canvas()

def on_trackbar_change(change):
  refresh_canvas()

for trackbar in [red_trackbar, green_trackbar, blue_trackbar, size_trackbar, opacity_trackbar]:
  trackbar.observe(on_trackbar_change, names='value')

draw_point_button.on_click(on_draw_point)
sample_stroke_button.on_click(on_sample_stroke)
eraser_button.on_click(on_eraser)
clear_button.on_click(on_clear)
undo_button.on_click(on_undo)
save_button.on_click(on_save)
clear_button.on_click(on_clear)

display(widgets.VBox([
    widgets.HTML('<h4>Five Paint Trackbars</h4>'),
    red_trackbar,
    green_trackbar,
    blue_trackbar,
    size_trackbar,
    opacity_trackbar,
    widgets.HTML('<h4>Drawing controls</h4>'),
    x_trackbar,
    y_trackbar,
    widgets.HBox([draw_point_button, sample_stroke_button, eraser_button]),
    widgets.HBox([undo_button, save_button, clear_button]),
    status,
    output_area
]))

refresh_canvas()